In [0]:
# Databricks notebook source
# MAGIC %run /Shared/param/constants

# COMMAND ----------

import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from collections import OrderedDict
import traceback
from distutils.util import strtobool
import pyspark.sql.functions as func
from pyspark.sql.functions import sha2, concat_ws, col, from_utc_timestamp,explode, concat, substring, lit
from pyspark.sql.types import DoubleType,IntegerType,TimestampType

# Variable settings
source_system = getArgument("SourceSystem")
dw_schema_name = getArgument("TargetSchema").lower() + '.'
table_name = getArgument("TableName")
target_table_name = table_name
source_filename = getArgument("SourceFileName")
target_date = getArgument("TargetDate")
from_date = getArgument("FromDate")
to_date = getArgument("ToDate")
manual_load = getArgument("ManualLoad")

if(manual_load == '1' and from_date != to_date):
  date = from_date + "-" + to_date
elif(manual_load == '1' and from_date == to_date):
  date = from_date
elif(manual_load == '0'):
  date = target_date

input_file = join(mount_point, getArgument("InputFolder") , date , getArgument("SourceFileName"))
data_folder_path = join(mount_point, getArgument("DataFolder"))
load_type = getArgument("LoadType")

# Sql DWH Setting
spark.conf.set("spark.databricks.sqldw.writeSemantics", dw_writeSemantics)
spark.conf.set(dw_key, dw_token)

# Validation Setting
target_row=0
validation_data_list=[]

def rows_validation(src_row,target_row):
  if(load_type == 'incr'):
    return valid_msg['skip']
 
  if(src_row == target_row):
    return valid_msg['pass']
  else:
    return valid_msg['fail']

# Define the columns of table [management].[BATCH_TABLE_PARTITION] 
def ins_partition_rec(col_list, data_list):
  dataDict = OrderedDict()
  for col, data in zip(col_list, data_list):
    dataDict[col]=data
  df = pd.DataFrame(data=[],columns=dataDict.keys())
  df = pd.concat([df, pd.DataFrame.from_dict([dataDict])])  
  spark_df = spark.createDataFrame(df, schema=lz_batch_partition_schema)
  spark_df.write.format("jdbc").mode(mode_a).option("driver", db_driver).option("url", db_jdbcUrl).option("dbtable", db_partition_table).save()

def get_timestamp(*, milisec=6):
  JST = timezone(timedelta(hours=+9), 'JST')
  if(milisec==6):
    return datetime.now(JST).strftime('%Y-%m-%d %H:%M:%S.%f')
  else:
    return datetime.now(JST).strftime('%Y-%m-%d %H:%M:%S.%f')[:-(6-milisec)]  

def init_valid_data(stepno):
  if(stepno=='2'):
    return [int(float(getArgument("StepId"))),int(float(getArgument("BatchId"))),getArgument("SourceSystem"),getArgument("TableName"),stepno,getArgument("steptype"),getArgument("PipelineId")]
  else:
    return [int(float(getArgument("StepId"))),int(float(getArgument("BatchId"))),getArgument("SourceSystem"),getArgument("TableName"),stepno,getArgument("steptype"),getArgument("PipelineId")]

try:
  # Reset validation info
  validation_data_list= init_valid_data(getArgument("stepno"))
  start_time=get_timestamp(milisec=3)
  
  # Load data to Sql DWH
  start = time.time()
  target_row=0
  print("Loading data to Sql Dwh started")
  
  # Import data to dataframe 
  df = spark.read.format(FILE_TYPE_CSV) \
                 .option("delimiter",CSV_DELIMITER) \
                 .option("header","true") \
                 .load(input_file)
  df.show(truncate=False)
  source_row = df.count()
  
  # Load data from dataframe to Sql Dwh table 
  df.write \
  .format(dw_format) \
  .option("url", dw_url) \
  .option("forwardSparkAzureStorageCredentials", dw_forwardSparkAzureStorageCredentials) \
  .option("dbTable", dw_schema_name + table_name) \
  .option("tempDir", dw_tempDir) \
  .option("maxStrLength", dw_maxStrLength) \
  .mode(mode_o) \
  .save() 
  print("Loading data to Sql Dwh completed")
  print("Loading data to Sql Dwh took %.2f seconds" % (time.time() - start),sep='')
  
  target_row = spark.read.format("jdbc") \
             .option("url", dw_url) \
             .option("dbTable", dw_schema_name + '[' + table_name + ']') \
             .load() \
             .count()
  

  validation_data_list = validation_data_list + [source_filename,start_time,get_timestamp(milisec=3),proc_status['success'],rows_validation(source_row,target_row),source_row,target_row,None,target_table_name]
  ins_partition_rec(lz_batch_partition_col_list,validation_data_list)
  
  
except Exception as e:
  validation_data_list = validation_data_list + [source_filename,start_time,get_timestamp(milisec=3),proc_status['failed'],valid_msg['fail'],0,0,None,target_table_name]
  ins_partition_rec(lz_batch_partition_col_list,validation_data_list)
  traceback.print_exc()
  raise(e)

# COMMAND ----------

# # Create dropdown widgets

# dbutils.widgets.text("TableName", "","") 
# dbutils.widgets.get("TableName")
# # dbutils.widgets.remove("TableName")

# dbutils.widgets.text("InputFolder", "","") 
# dbutils.widgets.get("InputFolder")
# # dbutils.widgets.remove("InputFolder")

# dbutils.widgets.text("DataFolder", "","") 
# dbutils.widgets.get("DataFolder")
# # dbutils.widgets.remove("DataFolder")

# dbutils.widgets.text("WorkFolder", "","") 
# dbutils.widgets.get("WorkFolder")
# # dbutils.widgets.remove("WorkFolder")

# dbutils.widgets.text("SourceSystem", "","") 
# dbutils.widgets.get("SourceSystem")
# # dbutils.widgets.remove("SourceSystem")

# dbutils.widgets.text("BatchId", "","") 
# dbutils.widgets.get("BatchId")
# # dbutils.widgets.remove("BatchId")

# dbutils.widgets.text("PipelineId", "","") 
# dbutils.widgets.get("PipelineId")
# # dbutils.widgets.remove("PipelineId")

# dbutils.widgets.text("StepId", "","") 
# dbutils.widgets.get("StepId")
# # dbutils.widgets.remove("StepId")

# dbutils.widgets.text("TargetSystem", "","") 
# dbutils.widgets.get("TargetSystem")
# # dbutils.widgets.remove("TargetSystem")

# dbutils.widgets.text("stepno", "","") 
# dbutils.widgets.get("stepno")
# # dbutils.widgets.remove("stepno")

# dbutils.widgets.text("steptype", "","") 
# dbutils.widgets.get("steptype")
# # dbutils.widgets.remove("steptype")

# dbutils.widgets.text("TargetDate", "","") 
# dbutils.widgets.get("TargetDate")
# # dbutils.widgets.remove("TargetDate")

# dbutils.widgets.text("LoadType", "","") 
# dbutils.widgets.get("LoadType")
# # dbutils.widgets.remove("LoadType")

# dbutils.widgets.text("TargetSchema", "","") 
# dbutils.widgets.get("TargetSchema")
# # dbutils.widgets.remove("TargetSchema")

# dbutils.widgets.text("EnvironmentInfo", "","") 
# dbutils.widgets.get("EnvironmentInfo")
# # dbutils.widgets.remove("EnvironmentInfo")

# dbutils.widgets.text("SourceFileName", "","") 
# dbutils.widgets.get("SourceFileName")
# # dbutils.widgets.remove("SourceFileName")

# dbutils.widgets.text("FromDate", "","") 
# dbutils.widgets.get("FromDate")
# dbutils.widgets.remove("FromDate")

# dbutils.widgets.text("ToDate", "","") 
# dbutils.widgets.get("ToDate")
# dbutils.widgets.remove("ToDate")

# dbutils.widgets.text("ToDate", "","") 
# dbutils.widgets.get("ToDate")
# dbutils.widgets.remove("ToDate")

# dbutils.widgets.text("ManualLoad", "","") 
# dbutils.widgets.get("ManualLoad")
# dbutils.widgets.remove("ManualLoad")